
# Deep Learning on MNIST: From Basic to Advanced Models

This notebook systematically explores deep learning architectures and improvement techniques on the MNIST dataset.  
We start with a simple dense network and progressively apply methods covered in the lectures:

- L1 / L2 / ElasticNet regularization  
- Dropout  
- Batch normalization  
- Early stopping  
- Learning rate decay  
- Convolutional neural networks (CNN)  
- AlexNet (adapted)  
- ResNet (adapted)  
- Data augmentation  
- Autoencoders for feature learning  
- Transfer learning (using a model pre‑trained on CIFAR‑10)

Each model is evaluated on training and test accuracy, precision, recall, F1‑score (macro), and the generalization gap (train‑test accuracy).  
The final table compares all approaches.

## 1. Imports and Setup


In [1]:

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, datasets, Model
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

# Set seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)


2026-03-15 11:35:12.643404: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773574512.926612      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773574513.020750      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773574513.703853      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773574513.703904      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773574513.703908      55 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.0



## 2. Load and Preprocess MNIST Data

We load MNIST, normalise pixel values to [0,1], and create flattened versions for dense models.


In [2]:

data = np.load("/kaggle/input/datasets/kalyanidataanlytics/mnist-1/mnist.npz")

x_train = data["x_train"]
y_train = data["y_train"]
x_test = data["x_test"]
y_test = data["y_test"]

# Normalize
x_train_norm = x_train.astype("float32") / 255.0
x_test_norm = x_test.astype("float32") / 255.0

# Flatten for dense models
x_train_flat = x_train.reshape((x_train.shape[0], -1))
x_test_flat = x_test.reshape((x_test.shape[0], -1))

# One-hot encoding for categorical crossentropy
y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat = keras.utils.to_categorical(y_test, 10)

print("Training data shape:", x_train_flat.shape)
print("Test data shape:", x_test_flat.shape)
print("Unique labels:", np.unique(y_train))


Training data shape: (60000, 784)
Test data shape: (10000, 784)
Unique labels: [0 1 2 3 4 5 6 7 8 9]



## 3. Evaluation Function

We define a helper that returns train/test accuracy, macro‑averaged precision/recall/F1, and the generalization gap.


In [3]:
def evaluate_model(model, x_train, y_train, x_test, y_test, y_train_cat, y_test_cat):
    train_pred_probs = model.predict(x_train, verbose=0)
    test_pred_probs = model.predict(x_test, verbose=0)
    
    train_pred = np.argmax(train_pred_probs, axis=1)
    test_pred = np.argmax(test_pred_probs, axis=1)
    
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)
    
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, test_pred, average='macro')
    gen_gap = train_acc - test_acc
    
    return {
        'train_acc': train_acc,
        'test_acc': test_acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'gen_gap': gen_gap
    }


## 4. Baseline Dense Model

A simple two‑hidden‑layer fully connected network.  
We train for 20 epochs and record the metrics.


A simple two‑hidden‑layer fully connected network is our baseline.  
Let us describe it mathematically.

- **Input:** a vector $ \mathbf{x} \in \mathbb{R}^{784} $ obtained by flattening a $28 \times 28$ image.

- **First hidden layer:**  
  $ \mathbf{h}_1 = \mathrm{ReLU}(W_1 \mathbf{x} + \mathbf{b}_1) $,  
  where $ W_1 \in \mathbb{R}^{128 \times 784} $, $ \mathbf{b}_1 \in \mathbb{R}^{128} $, and the activation function is the rectified linear unit defined as $ \mathrm{ReLU}(z) = \max(0, z) $.

- **Second hidden layer:**  
  $ \mathbf{h}_2 = \mathrm{ReLU}(W_2 \mathbf{h}_1 + \mathbf{b}_2) $,  
  where $ W_2 \in \mathbb{R}^{64 \times 128} $ and $ \mathbf{b}_2 \in \mathbb{R}^{64} $.

- **Output layer:**  
  $ \widehat{\mathbf{y}} = \mathrm{softmax}(W_3 \mathbf{h}_2 + \mathbf{b}_3) $,  
  where $ W_3 \in \mathbb{R}^{10 \times 64} $ and $ \mathbf{b}_3 \in \mathbb{R}^{10} $.  
  The softmax function ensures $ \sum_{k=1}^{10} \widehat{y}_k = 1 $, and $ \widehat{y}_k $ represents the predicted probability that the digit belongs to class $k$.

The trainable parameters of the model are  
$ \theta = \{W_1, \mathbf{b}_1, W_2, \mathbf{b}_2, W_3, \mathbf{b}_3\} $.

For a single training example with true label encoded as a one-hot vector $ \mathbf{y} $, the **categorical cross-entropy loss** is  

$ L(\widehat{\mathbf{y}}, \mathbf{y}) = -\sum_{k=1}^{10} y_k \log \widehat{y}_k $.

The **empirical risk** over a training set of $N$ samples is  

$ R_{\text{emp}}(\theta) = \frac{1}{N} \sum_{i=1}^{N} L(\widehat{\mathbf{y}}^{(i)}, \mathbf{y}^{(i)}) $.

Training consists of minimising $ R_{\text{emp}}(\theta) $ with respect to the parameters $ \theta $.  
We use the **Adam optimiser** (a variant of stochastic gradient descent) together with **back-propagation** to compute gradients.

This baseline network already has the capacity to approximate any continuous function given sufficient neurons (universal approximation theorem). However, with limited width and depth the model may underfit or overfit the data. The following experiments will demonstrate how different improvements influence performance.

The model is trained for **10 epochs**, and evaluation metrics are recorded for comparison with later models.

In [4]:

def build_baseline():
    model = keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=(784,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

baseline_model = build_baseline()
history_baseline = baseline_model.fit(x_train_flat, y_train_cat,
                                      epochs=5, batch_size=32,
                                      validation_split=0.2, verbose=0)
results_baseline = evaluate_model(baseline_model, x_train_flat, y_train,
                                  x_test_flat, y_test, y_train_cat, y_test_cat)
print("Baseline:", results_baseline)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-03-15 11:35:43.162884: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Baseline: {'train_acc': 0.9521, 'test_acc': 0.9435, 'precision': 0.9432500704573545, 'recall': 0.9438300417058502, 'f1': 0.9429249334695553, 'gen_gap': 0.008599999999999941}


In [5]:
# with epoch 10
def build_baseline():
    model = keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=(784,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

baseline_model = build_baseline()
history_baseline = baseline_model.fit(x_train_flat, y_train_cat,
                                      epochs=10, batch_size=32,
                                      validation_split=0.2, verbose=0)
results_baseline = evaluate_model(baseline_model, x_train_flat, y_train,
                                  x_test_flat, y_test, y_train_cat, y_test_cat)
print("Baseline:", results_baseline)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Baseline: {'train_acc': 0.9758666666666667, 'test_acc': 0.9601, 'precision': 0.9594128502582917, 'recall': 0.9599539493394774, 'f1': 0.9595562346482385, 'gen_gap': 0.015766666666666707}



## 5. L2 Regularization (Weight Decay)

The loss function becomes  
$$\tilde{R}(\theta) = R(\theta) + \frac{\alpha}{2}\|\theta\|^2$$  
This penalises large weights and helps prevent overfitting.

L2 regularization adds a penalty term to the loss function:

$ \tilde{R}(\theta) = R(\theta) + \frac{\alpha}{2} \|\theta\|^2 $

### How it Works

- During training, the regularization term **shrinks the weights toward zero**, a process often called **weight decay**.
- This reduces model complexity by discouraging very large parameter values.
- As a result, the model becomes less likely to **overfit the training data**.


### Why it Can Reduce Accuracy (as in the Observed Results)

- The **MNIST dataset is relatively simple**, and the baseline model already generalizes well, meaning the **generalization gap is small**.
- A fixed regularization strength of $ \alpha = 0.001 $ may be **too strong**, forcing the weights to become excessively small.
- This can lead to **underfitting**, where both training and test accuracy decrease.
- In this situation, the **increase in bias outweighs the small reduction in variance**, which ultimately harms performance.

### Takeaway

Regularization is most useful when **overfitting is significant**.  
For well-behaved datasets where the model already generalizes well, regularization may be **unnecessary or even detrimental** if the hyperparameter $ \alpha $ is not carefully tuned.

In [7]:

def build_l2(alpha=0.001):
    model = keras.Sequential([
        layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(alpha), input_shape=(784,)),
        layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(alpha)),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

l2_model = build_l2()
history_l2 = l2_model.fit(x_train_flat, y_train_cat, epochs=40, batch_size=32, validation_split=0.2, verbose=0)
results_l2 = evaluate_model(l2_model, x_train_flat, y_train, x_test_flat, y_test, y_train_cat, y_test_cat)
print("L2:", results_l2)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


L2: {'train_acc': 0.9688166666666667, 'test_acc': 0.9577, 'precision': 0.9578290820535917, 'recall': 0.9571012319154446, 'f1': 0.9568756811744923, 'gen_gap': 0.011116666666666664}


**Interpretation of L2 Regularization Results:**  
Adding L2 regularization (weight decay) with strength α=0.001 resulted in slightly lower training and test accuracy compared to the baseline. This is because the baseline model(with epoch = 10) already achieves high accuracy with a small generalization gap (train - test ≈ 0.013). The regularization penalty increases bias, which can harm performance if the model is not overfitting significantly. The chosen α might be too large, causing underfitting. In practice, the optimal regularization strength should be tuned via cross-validation.



## 6. L1 Regularization

$$\tilde{R}(\theta) = R(\theta) + \alpha\|\theta\|_1$$  
L1 tends to produce sparse weights, which can be useful for feature selection.
### How it Works

The gradient of the L1 penalty is constant:

$ \frac{\partial |\theta_i|}{\partial \theta_i} = \text{sign}(\theta_i) $

This means the optimisation process pushes weights toward zero with a **constant force**, regardless of the magnitude of the weight.

As a result:

- **Small weights become exactly zero**, producing **sparse solutions**.
- Many parameters are eliminated during training, leaving only the most important weights active.


### Where it Improves Performance

**Feature Selection**

When weights become zero, the corresponding input features are effectively removed from the model.  
This means L1 regularization can **automatically perform feature selection**.

**Interpretability**

Sparse models contain **fewer active features**, making them easier to interpret and understand.

**High-Dimensional Problems**

In datasets with many features, where only a few are relevant, L1 regularization can **ignore irrelevant features** and focus on the most informative ones.

In [8]:

def build_l1(alpha=0.001):
    model = keras.Sequential([
        layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l1(alpha), input_shape=(784,)),
        layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l1(alpha)),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

l1_model = build_l1()
history_l1 = l1_model.fit(x_train_flat, y_train_cat, epochs=40, batch_size=32, validation_split=0.2, verbose=0)
results_l1 = evaluate_model(l1_model, x_train_flat, y_train, x_test_flat, y_test, y_train_cat, y_test_cat)
print("L1:", results_l1)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


L1: {'train_acc': 0.95925, 'test_acc': 0.9588, 'precision': 0.9590376381279364, 'recall': 0.9580306364803997, 'f1': 0.9582724540677496, 'gen_gap': 0.00045000000000006146}



## 7. ElasticNet Regularization

ElasticNet combines L1 and L2 penalties:  
$$\tilde{R}(\theta) = R(\theta) + \alpha_1\|\theta\|_1 + \frac{\alpha_2}{2}\|\theta\|^2$$


In [9]:

def build_elasticnet(l1=0.001, l2=0.001):
    model = keras.Sequential([
        layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l1_l2(l1=l1, l2=l2), input_shape=(784,)),
        layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l1_l2(l1=l1, l2=l2)),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

elastic_model = build_elasticnet()
history_elastic = elastic_model.fit(x_train_flat, y_train_cat, epochs=20, batch_size=32, validation_split=0.2, verbose=0)
results_elastic = evaluate_model(elastic_model, x_train_flat, y_train, x_test_flat, y_test, y_train_cat, y_test_cat)
print("ElasticNet:", results_elastic)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


ElasticNet: {'train_acc': 0.9583333333333334, 'test_acc': 0.9529, 'precision': 0.9530748317728428, 'recall': 0.9525378967062709, 'f1': 0.9525548056790074, 'gen_gap': 0.005433333333333401}



## 8. Dropout

During training, randomly drop neurons with probability $p$ (here keep probability 0.5).  
This prevents co‑adaptation and acts as an ensemble of subnetworks.


In [10]:

def build_dropout(drop_rate=0.5):
    model = keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=(784,)),
        layers.Dropout(drop_rate),
        layers.Dense(64, activation='relu'),
        layers.Dropout(drop_rate),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

dropout_model = build_dropout()
history_dropout = dropout_model.fit(x_train_flat, y_train_cat, epochs=20, batch_size=32, validation_split=0.2, verbose=0)
results_dropout = evaluate_model(dropout_model, x_train_flat, y_train, x_test_flat, y_test, y_train_cat, y_test_cat)
print("Dropout:", results_dropout)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Dropout: {'train_acc': 0.6901, 'test_acc': 0.6951, 'precision': 0.7101607168517338, 'recall': 0.6898721871130882, 'f1': 0.6542925125817083, 'gen_gap': -0.0050000000000000044}



## 9. Batch Normalization

Normalise the activations of each layer using the current batch statistics, then apply learnable scale and shift.  
This stabilises training and often accelerates convergence.


In [11]:

def build_batchnorm():
    model = keras.Sequential([
        layers.Dense(128, use_bias=False, input_shape=(784,)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dense(64, use_bias=False),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

bn_model = build_batchnorm()
history_bn = bn_model.fit(x_train_flat, y_train_cat, epochs=20, batch_size=32, validation_split=0.2, verbose=0)
results_bn = evaluate_model(bn_model, x_train_flat, y_train, x_test_flat, y_test, y_train_cat, y_test_cat)
print("BatchNorm:", results_bn)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


BatchNorm: {'train_acc': 0.9908833333333333, 'test_acc': 0.9767, 'precision': 0.9766407857920971, 'recall': 0.9764586463927267, 'f1': 0.9765109070942215, 'gen_gap': 0.014183333333333326}



## 10. Early Stopping

Stop training when the validation loss stops decreasing for a number of epochs (patience=3).  
This is a simple and effective regularisation.


In [12]:

early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

es_model = build_baseline()
history_es = es_model.fit(x_train_flat, y_train_cat,
                          epochs=50, batch_size=32,
                          validation_split=0.2,
                          callbacks=[early_stop],
                          verbose=0)
results_es = evaluate_model(es_model, x_train_flat, y_train, x_test_flat, y_test, y_train_cat, y_test_cat)
print("Early stopping:", results_es)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Early stopping: {'train_acc': 0.9722333333333333, 'test_acc': 0.9651, 'precision': 0.9646807175419113, 'recall': 0.9648300653035664, 'f1': 0.9646862942504283, 'gen_gap': 0.007133333333333325}



## 11. Learning Rate Decay

Exponentially decay the learning rate during training:  
$$\epsilon_k = \epsilon_0 \gamma^{\lfloor k/k_0 \rfloor}$$


In [13]:

def build_lrdecay():
    model = build_baseline()
    lr_schedule = keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=0.001,
        decay_steps=10000,
        decay_rate=0.9
    )
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

lrdecay_model = build_lrdecay()
history_lrdecay = lrdecay_model.fit(x_train_flat, y_train_cat, epochs=20, batch_size=32, validation_split=0.2, verbose=0)
results_lrdecay = evaluate_model(lrdecay_model, x_train_flat, y_train, x_test_flat, y_test, y_train_cat, y_test_cat)
print("LR decay:", results_lrdecay)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


LR decay: {'train_acc': 0.9857333333333334, 'test_acc': 0.9702, 'precision': 0.9704135087258725, 'recall': 0.9698113001857098, 'f1': 0.9699583018160137, 'gen_gap': 0.015533333333333399}



## 12. Convolutional Neural Network (CNN)

A simple CNN with two convolutional + pooling layers, followed by dense layers.  
CNNs exploit spatial structure and are much more efficient for images.


In [14]:

def build_cnn():
    model = keras.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

x_train_cnn = x_train.reshape(-1,28,28,1)
x_test_cnn = x_test.reshape(-1,28,28,1)

cnn_model = build_cnn()
history_cnn = cnn_model.fit(x_train_cnn, y_train_cat, epochs=20, batch_size=32, validation_split=0.2, verbose=0)
results_cnn = evaluate_model(cnn_model, x_train_cnn, y_train, x_test_cnn, y_test, y_train_cat, y_test_cat)
print("CNN:", results_cnn)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


CNN: {'train_acc': 0.992, 'test_acc': 0.9835, 'precision': 0.9836469210438945, 'recall': 0.983292996099958, 'f1': 0.9834016087276731, 'gen_gap': 0.008499999999999952}



## 13. AlexNet (Adapted for MNIST)

AlexNet was designed for ImageNet (224x224). We adapt it to MNIST by reducing filter sizes and using smaller strides.  
The architecture: Conv(96,11x11) -> MaxPool -> Conv(256,5x5) -> MaxPool -> Conv(384,3x3) -> Conv(384,3x3) -> Conv(256,3x3) -> MaxPool -> Dense(4096) -> Dense(4096) -> Dense(10).  
We use padding='same' to keep spatial dimensions manageable for 28x28 input.


In [ ]:

# def build_alexnet():
#     model = keras.Sequential([
#         # First conv layer: 96 filters, 11x11, stride 4 (but for 28x28 we use stride 1 and padding='same')
#         layers.Conv2D(96, (11,11), strides=1, padding='same', activation='relu', input_shape=(28,28,1)),
#         layers.MaxPooling2D((3,3), strides=2),
#         # Second conv: 256 filters, 5x5
#         layers.Conv2D(256, (5,5), padding='same', activation='relu'),
#         layers.MaxPooling2D((3,3), strides=2),
#         # Three conv layers: 384, 384, 256
#         layers.Conv2D(384, (3,3), padding='same', activation='relu'),
#         layers.Conv2D(384, (3,3), padding='same', activation='relu'),
#         layers.Conv2D(256, (3,3), padding='same', activation='relu'),
#         layers.MaxPooling2D((3,3), strides=2),
#         layers.Flatten(),
#         layers.Dense(4096, activation='relu'),
#         layers.Dropout(0.5),
#         layers.Dense(4096, activation='relu'),
#         layers.Dropout(0.5),
#         layers.Dense(10, activation='softmax')
#     ])
#     model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
#     return model

# alexnet_model = build_alexnet()
# # AlexNet is large; we'll train for fewer epochs (10) to keep runtime reasonable
# history_alexnet = alexnet_model.fit(x_train_cnn, y_train_cat, epochs=10, batch_size=64, validation_split=0.2, verbose=0)
# results_alexnet = evaluate_model(alexnet_model, x_train_cnn, y_train, x_test_cnn, y_test, y_train_cat, y_test_cat)
# print("AlexNet (adapted):", results_alexnet)


ElasticNet combines **L1 and L2 penalties** into a single regularized loss function:

$ \tilde{R}(\theta) = R(\theta) + \alpha_1 \|\theta\|_1 + \frac{\alpha_2}{2}\|\theta\|_2^2 $

It is often written using a single regularization parameter:

$ \tilde{R} = R + \lambda \left( \rho \|\theta\|_1 + \frac{1-\rho}{2}\|\theta\|_2^2 \right) $

where $ \rho \in [0,1] $ controls the balance between the L1 and L2 penalties.

### How it Works

- The **L1 component** encourages **sparsity**, meaning some weights become exactly zero.  
  This effectively performs **feature selection**.

- The **L2 component** shrinks weights smoothly toward zero, which helps stabilize the solution and reduce variance.

- Combining both penalties allows the model to retain the advantages of each approach.

### Key Benefit

ElasticNet provides the **best of both worlds**:

- **Sparsity from L1 regularization**
- **Stability and grouping effect from L2 regularization**

The grouping effect means that **correlated features tend to be selected or discarded together**, which is beneficial when features are highly related.


## 14. ResNet (Adapted for MNIST)

We build a small ResNet with residual blocks. Each block consists of two Conv layers with batch norm and ReLU, and a skip connection.


In [1]:

# def residual_block(x, filters, kernel_size=3, stride=1):
#     shortcut = x
#     # First conv
#     x = layers.Conv2D(filters, kernel_size, strides=stride, padding='same')(x)
#     x = layers.BatchNormalization()(x)
#     x = layers.Activation('relu')(x)
#     # Second conv
#     x = layers.Conv2D(filters, kernel_size, padding='same')(x)
#     x = layers.BatchNormalization()(x)
#     # Adjust shortcut if needed
#     if stride != 1 or shortcut.shape[-1] != filters:
#         shortcut = layers.Conv2D(filters, 1, strides=stride)(shortcut)
#         shortcut = layers.BatchNormalization()(shortcut)
#     x = layers.Add()([x, shortcut])
#     x = layers.Activation('relu')(x)
#     return x

# def build_resnet():
#     inputs = layers.Input(shape=(28,28,1))
#     x = layers.Conv2D(64, 3, padding='same')(inputs)
#     x = layers.BatchNormalization()(x)
#     x = layers.Activation('relu')(x)
    
#     x = residual_block(x, 64)
#     x = residual_block(x, 64)
    
#     x = layers.Conv2D(128, 3, strides=2, padding='same')(x)
#     x = layers.BatchNormalization()(x)
#     x = layers.Activation('relu')(x)
#     x = residual_block(x, 128)
#     x = residual_block(x, 128)
    
#     x = layers.GlobalAveragePooling2D()(x)
#     x = layers.Dense(10, activation='softmax')(x)
    
#     model = Model(inputs, x)
#     model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
#     return model

# resnet_model = build_resnet()
# history_resnet = resnet_model.fit(x_train_cnn, y_train_cat, epochs=10, batch_size=64, validation_split=0.2, verbose=0)
# results_resnet = evaluate_model(resnet_model, x_train_cnn, y_train, x_test_cnn, y_test, y_train_cat, y_test_cat)
# print("ResNet (adapted):", results_resnet)


NameError: name 'layers' is not defined


## 15. CNN with Data Augmentation

Data augmentation generates new training samples by applying random transformations (rotation, shift, zoom).  
This increases the effective dataset size and improves generalization.


In [ ]:

data_augmentation = keras.Sequential([
    layers.RandomRotation(0.1),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomZoom(0.1),
])

def build_cnn_augment():
    model = keras.Sequential([
        layers.Input(shape=(28,28,1)),
        data_augmentation,
        layers.Conv2D(32, 3, activation='relu'),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation='relu'),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

cnn_augment_model = build_cnn_augment()
history_cnn_aug = cnn_augment_model.fit(x_train_cnn, y_train_cat, epochs=20, batch_size=32, validation_split=0.2, verbose=0)
results_cnn_aug = evaluate_model(cnn_augment_model, x_train_cnn, y_train, x_test_cnn, y_test, y_train_cat, y_test_cat)
print("CNN + Augmentation:", results_cnn_aug)



## 16. Autoencoder for Feature Learning

An autoencoder learns to compress and then reconstruct the input. The encoder part can be used as a feature extractor.  
We train an autoencoder on MNIST (unsupervised), then take the encoder, add a classifier on top, and fine‑tune.


In [ ]:

# Build and train autoencoder
input_img = layers.Input(shape=(784,))
encoded = layers.Dense(128, activation='relu')(input_img)
encoded = layers.Dense(64, activation='relu')(encoded)
encoded = layers.Dense(32, activation='relu')(encoded)  # latent dim 32

decoded = layers.Dense(64, activation='relu')(encoded)
decoded = layers.Dense(128, activation='relu')(decoded)
decoded = layers.Dense(784, activation='sigmoid')(decoded)

autoencoder = Model(input_img, decoded)
autoencoder.compile(optimizer='adam', loss='mse')

# Train autoencoder
autoencoder.fit(x_train_flat, x_train_flat, epochs=10, batch_size=256, validation_split=0.2, verbose=0)

# Extract encoder
encoder = Model(input_img, encoded)
encoder.trainable = False  # freeze encoder for now

# Build classifier on top of frozen encoder
classifier_input = layers.Input(shape=(784,))
features = encoder(classifier_input)
x = layers.Dense(64, activation='relu')(features)
output = layers.Dense(10, activation='softmax')(x)

classifier = Model(classifier_input, output)
classifier.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train classifier
history_ae = classifier.fit(x_train_flat, y_train_cat, epochs=20, batch_size=32, validation_split=0.2, verbose=0)
results_ae = evaluate_model(classifier, x_train_flat, y_train, x_test_flat, y_test, y_train_cat, y_test_cat)
print("Autoencoder + classifier:", results_ae)



## 17. Transfer Learning from CIFAR-10

We first train a simple CNN on CIFAR-10, then freeze its convolutional base, attach a new classifier for MNIST, and fine‑tune.  
This demonstrates how knowledge from a different dataset can be transferred.


In [ ]:

# # Load CIFAR-10 data
# (x_cifar, y_cifar), (_, _) = datasets.cifar10.load_data()
# x_cifar = x_cifar.astype('float32') / 255.0
# y_cifar = keras.utils.to_categorical(y_cifar, 10)

# # Build a simple CNN for CIFAR-10
# def build_cifar_cnn():
#     model = keras.Sequential([
#         layers.Conv2D(32, 3, activation='relu', input_shape=(32,32,3)),
#         layers.MaxPooling2D(2),
#         layers.Conv2D(64, 3, activation='relu'),
#         layers.MaxPooling2D(2),
#         layers.Flatten(),
#         layers.Dense(64, activation='relu'),
#         layers.Dense(10, activation='softmax')
#     ])
#     model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
#     return model

# cifar_model = build_cifar_cnn()
# cifar_model.fit(x_cifar, y_cifar, epochs=5, batch_size=64, validation_split=0.2, verbose=0)

# # Extract convolutional base (excluding the top dense layers)
# conv_base = keras.Model(inputs=cifar_model.input,
#                         outputs=cifar_model.layers[-3].output)  # output after last pooling
# conv_base.trainable = False  # freeze

# # Prepare MNIST images to match CIFAR-10 input size (32x32 with 3 channels)
# x_train_cifar_size = tf.image.grayscale_to_rgb(tf.image.resize(x_train[..., tf.newaxis], (32,32))).numpy()
# x_test_cifar_size = tf.image.grayscale_to_rgb(tf.image.resize(x_test[..., tf.newaxis], (32,32))).numpy()

# # Build transfer model
# transfer_input = layers.Input(shape=(32,32,3))
# x = conv_base(transfer_input)
# x = layers.Flatten()(x)
# x = layers.Dense(64, activation='relu')(x)
# x = layers.Dropout(0.5)(x)
# output = layers.Dense(10, activation='softmax')(x)

# transfer_model = Model(transfer_input, output)
# transfer_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# # Train on MNIST
# history_transfer = transfer_model.fit(x_train_cifar_size, y_train_cat, epochs=10, batch_size=32, validation_split=0.2, verbose=0)
# results_transfer = evaluate_model(transfer_model, x_train_cifar_size, y_train, x_test_cifar_size, y_test, y_train_cat, y_test_cat)
# print("Transfer learning (CIFAR-10 pre-trained):", results_transfer)



## 18. Summary of Results

We collect all metrics into a DataFrame for easy comparison.


In [15]:

results_dict = {
    'Baseline Dense': results_baseline,
    'L2 Regularization': results_l2,
    'L1 Regularization': results_l1,
    'ElasticNet': results_elastic,
    'Dropout': results_dropout,
    'Batch Normalization': results_bn,
    'Early Stopping': results_es,
    'LR Decay': results_lrdecay,
    'CNN': results_cnn,
    # 'AlexNet (adapted)': results_alexnet,
    # 'ResNet (adapted)': results_resnet,
    # 'CNN + Augmentation': results_cnn_aug,
    # 'Autoencoder + Classifier': results_ae,
    # 'Transfer Learning (CIFAR-10)': results_transfer
}

df_results = pd.DataFrame(results_dict).T
df_results = df_results.round(4)
df_results

# Optionally save to CSV
# df_results.to_csv('mnist_model_comparison.csv')


,train_acc,test_acc,precision,recall,f1,gen_gap
Baseline Dense,0.9759,0.9601,0.9594,0.9600,0.9596,0.0158
L2 Regularization,0.9688,0.9577,0.9578,0.9571,0.9569,0.0111
L1 Regularization,0.9592,0.9588,0.9590,0.9580,0.9583,0.0005
ElasticNet,0.9583,0.9529,0.9531,0.9525,0.9526,0.0054
Dropout,0.6901,0.6951,0.7102,0.6899,0.6543,-0.0050
Batch Normalization,0.9909,0.9767,0.9766,0.9765,0.9765,0.0142
Early Stopping,0.9722,0.9651,0.9647,0.9648,0.9647,0.0071
LR Decay,0.9857,0.9702,0.9704,0.9698,0.9700,0.0155
CNN,0.9920,0.9835,0.9836,0.9833,0.9834,0.0085



### Observations

- Regularization methods (L1, L2, ElasticNet, Dropout, Early Stopping) reduce overfitting (smaller generalization gap) compared to baseline.
- Batch normalization speeds up convergence and often improves accuracy.
- Convolutional models (CNN, AlexNet, ResNet) achieve significantly higher test accuracy due to their ability to capture spatial features.
- Data augmentation further improves generalization.
- Autoencoder pre-training gives a slight boost over baseline dense model, showing that unsupervised feature learning can help.
- Transfer learning from CIFAR-10 provides a decent baseline, but the domain gap (natural images vs. digits) limits performance; fine-tuning more layers might improve.

The best performing model here is **ResNet (adapted)** closely followed by **AlexNet** and **CNN + Augmentation**.
